# 跨境电商销量预测 - 模型评估

评估指标：MAE, RMSE, MAPE, WAPE, 分位数损失

In [ ]:
import sys
sys.path.append("code_evaluate")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from evaluate import evaluate_forecast, evaluate_by_period, print_summary

## 1. 加载数据

In [ ]:
# 加载预测结果
forecast_df = pd.read_csv("forecast_US_20260130.csv", parse_dates=["date"])

# 加载实际销量
# actual_df = pd.read_csv("actual_sales.csv", parse_dates=["date"])

# ========== 示例数据 ==========
np.random.seed(42)
actual_df = forecast_df.copy()
actual_df["sales_quantity"] = (actual_df["forecast"] * np.random.uniform(0.8, 1.2, len(actual_df))).astype(int)

print(f"预测记录: {len(forecast_df)}")
print(f"实际记录: {len(actual_df)}")

## 2. 按产品评估

In [ ]:
metrics_df = evaluate_forecast(actual_df, forecast_df)
print_summary(metrics_df)
metrics_df

## 3. 按时间段评估

In [ ]:
period_metrics = evaluate_by_period(actual_df, forecast_df)
period_metrics

## 4. 可视化

In [ ]:
def plot_actual_vs_forecast(actual_df, forecast_df, asin):
    actual = actual_df[actual_df["asin"] == asin].set_index("date")["sales_quantity"]
    pred = forecast_df[forecast_df["asin"] == asin].set_index("date")
    
    plt.figure(figsize=(12, 5))
    plt.plot(actual.index, actual.values, "o-", label="实际销量", color="#2E86AB")
    plt.plot(pred.index, pred["forecast"], "s--", label="预测销量", color="#E94F37")
    
    if "lower_10" in pred.columns:
        plt.fill_between(pred.index, pred["lower_10"], pred["upper_90"],
                         alpha=0.2, color="#E94F37", label="90% 置信区间")
    
    plt.title(f"产品 {asin} - 实际 vs 预测")
    plt.xlabel("日期")
    plt.ylabel("销量")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

best_asin = metrics_df.loc[metrics_df["WAPE"].idxmin(), "asin"]
worst_asin = metrics_df.loc[metrics_df["WAPE"].idxmax(), "asin"]

print("最佳产品:")
plot_actual_vs_forecast(actual_df, forecast_df, best_asin)

print("最差产品:")
plot_actual_vs_forecast(actual_df, forecast_df, worst_asin)

In [ ]:
def plot_error_distribution(metrics_df):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    axes[0].hist(metrics_df["WAPE"], bins=20, color="#2E86AB", edgecolor="white")
    axes[0].axvline(metrics_df["WAPE"].mean(), color="red", linestyle="--", label=f"均值: {metrics_df['WAPE'].mean():.1f}%")
    axes[0].set_xlabel("WAPE (%)")
    axes[0].set_ylabel("产品数量")
    axes[0].set_title("WAPE 分布")
    axes[0].legend()
    
    axes[1].hist(metrics_df["Bias"], bins=20, color="#A23B72", edgecolor="white")
    axes[1].axvline(0, color="black", linestyle="-", alpha=0.5)
    axes[1].axvline(metrics_df["Bias"].mean(), color="red", linestyle="--", label=f"均值: {metrics_df['Bias'].mean():.1f}")
    axes[1].set_xlabel("Bias (正=高估, 负=低估)")
    axes[1].set_ylabel("产品数量")
    axes[1].set_title("预测偏差分布")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

plot_error_distribution(metrics_df)

## 5. 业务影响分析

In [ ]:
def calc_business_impact(actual_df, forecast_df, unit_cost=10, holding_cost_rate=0.02):
    merged = forecast_df.merge(
        actual_df[["asin", "date", "sales_quantity"]],
        on=["asin", "date"],
        how="inner"
    )
    
    merged["error"] = merged["forecast"] - merged["sales_quantity"]
    
    overstock = merged[merged["error"] > 0]["error"].sum()
    overstock_cost = overstock * unit_cost * holding_cost_rate
    
    stockout = abs(merged[merged["error"] < 0]["error"].sum())
    stockout_cost = stockout * unit_cost * 0.3
    
    print("=" * 50)
    print("业务影响分析")
    print("=" * 50)
    print(f"预测高估总量: {overstock:,.0f} 件")
    print(f"库存积压成本: ${overstock_cost:,.2f}")
    print(f"预测低估总量: {stockout:,.0f} 件")
    print(f"缺货损失:     ${stockout_cost:,.2f}")
    print(f"总影响成本:   ${overstock_cost + stockout_cost:,.2f}")
    print("=" * 50)

calc_business_impact(actual_df, forecast_df)

## 6. 导出报告

In [ ]:
from datetime import datetime

output_file = f"evaluation_report_{datetime.now().strftime('%Y%m%d')}.csv"
metrics_df.to_csv(output_file, index=False)
print(f"评估报告已保存: {output_file}")